# Silicon MZM with a GSG probe

Schematic reconstruction after Reed et al., Nat. Photonics 2010
([doi:10.1038/nphoton.2010.179](https://doi.org/10.1038/nphoton.2010.179)).

Each GSG landing is its own square `ElectricalPad`. The MZM
electrical ports leave as `path_type="L"` traces at different
widths and west lengths, then jog on stacked rails to the pads
above so the nets do not cross the optical input.


In [1]:
from optodraw import Scene

def half_size(component):
    probe = Scene(title="", width=2000, height=2000, platform="on_chip")
    probe.add(component)
    x0, y0, x1, y1 = probe.content_bounds()
    return 0.5 * (x1 - x0), 0.5 * (y1 - y0)


In [2]:
def dangling_optical(scene, connected):
    used = {(id(comp), name) for comp, name in connected}
    leftover = []
    for component in scene.walk():
        ports = getattr(component, "ports", None)
        if not callable(ports):
            continue
        for name, port in ports().items():
            if getattr(port, "kind", "optical") in ("electrical", "fiber"):   # a fibre port is the off-chip side
                continue
            if name.startswith(("e", "h", "gnd", "sig")):
                continue
            if (id(component), name) in used:
                continue
            label = getattr(component, "label", None) or component.__class__.__name__
            leftover.append(f"{label}:{name}")
    return leftover


In [3]:
from optodraw import Scene, VerticalGratingCoupler, MZM, OnChipPhotodetector, ElectricalPad
from optodraw.layout import chain_ports

mzm_hh = half_size(MZM(drive_style="gsg", label="MZM"))[1]
pad_hw, pad_hh = half_size(ElectricalPad(width=16, height=16, port_side="bottom", label="S"))
PITCH = 2.0 * pad_hw + 6.0
HOP = 20.0
print("MZM half height", mzm_hh, "pad pitch", PITCH)


MZM half height 15.39 pad pitch 38.0


In [4]:
scene = Scene(title="MZM with GSG probe (after Reed 2010)", width=400, height=200, platform="on_chip")
gc = scene.add(VerticalGratingCoupler(center=(0, 0), angle=180, label="in"))
mzm = scene.add(MZM(drive_style="gsg", label="MZM"))
pd = scene.add(OnChipPhotodetector(label="PD"))
chain_ports([(gc, "o1", mzm, "o1"), (mzm, "o3", pd, "o1")], gap=HOP, grid=0)
mzm_top = float(mzm.transform.position.y) + mzm_hh
y_pad = mzm_top + 20.0 + pad_hh
x_mid = float(mzm.transform.position.x)
pads = {
    name: scene.add(ElectricalPad(center=(x_mid + (i - 1) * PITCH, y_pad), width=16, height=16, port_side="bottom", label=name))
    for i, name in enumerate(("G2", "S", "G1"))
}
scene.route((gc, "o1"), (mzm, "o1"), kind="waveguide", path_type="smooth")
scene.route((mzm, "o3"), (pd, "o1"), kind="waveguide", path_type="smooth")
for port, name, width, west, rail in (
    ("gnd1", "G1", 2.4, 6.0, mzm_top + 9.0),
    ("sig", "S", 1.4, 12.0, mzm_top + 15.0),
    ("gnd2", "G2", 2.4, 18.0, mzm_top + 3.0),
):
    scene.route(
        (mzm, port),
        (pads[name], "e1"),
        kind="electrical",
        path_type="L",
        start_straight_length=west,
        end_straight_length=3,
        width=width,
        steps=[{"y": rail}],
    )
leftover = dangling_optical(scene, [(gc, "o1"), (mzm, "o1"), (mzm, "o3"), (pd, "o1")])
print("unconnected", leftover)
scene.fit_frame(margin=16)


unconnected ['MZM:o2', 'MZM:o4']


In [5]:
print("bounds", scene.content_bounds())
scene.preview(show_3d=False)


bounds (np.float64(-20.7975), np.float64(-15.390000000000008), np.float64(235.0), np.float64(67.38999999999999))


ScenePreview('MZM with GSG probe (after Reed 2010)', 900x520, 3d=off)